# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkharche01/Internship-starter-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines and verifies a simple modeling contract for Search Intelligence data.

The work is intentionally based on information available at the decision point. February 2026 is used as the feature window and March 2026 is used as the future outcome window.

The goal is to keep the feature set small, observable, and free from future-data leakage.

## 1. Unit of analysis + time window

### Unit of analysis

One row in my modeling dataset represents **one content item for one client at one decision point**.

The source performance table is daily, so I aggregate daily observations to the client × content level for the decision point.

### Feature window

I use **February 2026** as the feature window.

The decision point is **February 28, 2026**. Every feature must be knowable by this date.

### Outcome window

I use **March 2026** as the outcome window.

March performance is observed after the decision point and is used only to define the future outcome.

### Prediction target

I will estimate whether a content item receives **zero measured Google Search Console clicks during March 2026**.

The future March click outcome is used as the label.

### Excluded

I deliberately exclude March performance variables from the final feature set because they are only known after the February 28 decision point. Including them would leak future outcome information into the model.

In [40]:
from huggingface_hub import HfApi

api = HfApi()

user = api.whoami()

print("Hugging Face login successful.")
print("Username:", user["name"])

Hugging Face login successful.
Username: mayur-0110


In [41]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem()

repo = "datasets/FlyRank/internship-warehouse"

files = fs.glob(
    f"{repo}/**",
    detail=False
)

print("Number of files found:", len(files))

Number of files found: 44


In [42]:
[x for x in files if "fact_content_daily_performance" in x]

['datasets/FlyRank/internship-warehouse/fact_content_daily_performance',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05',
 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05

In [43]:
[x for x in files if "dim_content" in x]

['datasets/FlyRank/internship-warehouse/dim_content.parquet']

In [44]:
[x for x in files if "fact_content_query_90d" in x]

['datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet']

In [45]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [46]:
from huggingface_hub import get_token

HF_TOKEN = get_token()

print("Hugging Face token available:", HF_TOKEN is not None)

Hugging Face token available: True


In [47]:
con = duckdb.connect()

con.execute("SET enable_progress_bar = false")

print("DuckDB connected.")

DuckDB connected.


In [48]:
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face access configured.")

Hugging Face access configured.


In [49]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"

CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Warehouse paths defined.")

Warehouse paths defined.


In [50]:
con.sql(f"""
SELECT *
FROM {FEB}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


In [51]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM {FEB}
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [53]:
schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [54]:
[c for c in schema["column_name"].tolist() if "date" in c.lower()]

['report_date']

## 2. Fields: feature / label / context / excluded

### Features

1. **February clicks** — total Google Search Console clicks observed during February 2026.
2. **February impressions** — total Google Search Console impressions observed during February 2026.
3. **February average position** — average Search Console position observed during February 2026.
4. **February content age** — age of the content as of February 28, 2026.
5. **February active days** — number of February days with available Search Console observations.

### Label

**March zero-click outcome** — whether the content item has zero measured Google Search Console clicks during March 2026.

The underlying March click total is used only to construct the future label. It is not included in the final feature set.

### Context

- **Client ID** — identifies the pseudonymized client.
- **Content ID** — identifies the pseudonymized content item.
- **Decision date** — February 28, 2026.

### Excluded

March performance variables are excluded from the final feature set because they are only known after the February decision point.

Client names, URLs, raw queries, and other identifying information are also excluded from notebook outputs.

## 3. Verify it with queries

The following three small queries verify the main claims in the contract.

The checks are performed on the February 2026 feature window rather than the final June `_sample` data.

The three checks cover:

1. Source grain
2. Row count and date span
3. Google Search Console availability

### Query 1 — Grain

The source daily table should contain one row per client, content item, and report date.

I will check whether any client × content × date combination occurs more than once.

If the query returns zero rows, the stated daily grain is supported by the observed February data.

In [55]:
# QUERY 1 — Grain check

grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {FEB}
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
ORDER BY row_count DESC
""").df()

grain_check

,client_hash_id,content_hash_id,report_date,row_count


### Query 1 result

The grain check returned no duplicate client × content × report-date combinations.

This supports the use of the daily source table as one observation per client, content item, and report date for the February 2026 feature window.

### Query 2 — Row count and date window

I will verify the number of February observations and the observed first and last report dates.

This checks that the feature window contains the expected February 2026 date range.

In [57]:
# QUERY 2 — February row count and date range

feb_window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}
""").df()

feb_window_check

,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


### Query 2 result

The row-count and date-range check confirms the number of February 2026 observations available in the selected source partition.

The observed date range is used as the actual evidence for the February feature window rather than assuming the window is complete without checking the data.

### Query 3 — Verify Google Search Console availability

Google Search Console data may not be available for every source observation.

I will calculate the number and percentage of rows where `gsc_data_available` is explicitly `TRUE`.

Unavailable observations are not treated as zero clicks or zero impressions.

In [58]:
# QUERY 3 — GSC availability

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS available_pct
FROM {FEB}
""").df()

availability_check

,total_rows,available_rows,available_pct
0,7355108,2621783,35.65


### Query 3 result

The availability check measures the share of February observations with Google Search Console data available.

I use `gsc_data_available IS TRUE` explicitly so that unavailable observations are distinguished from measured zero performance.

### Verification conclusion

The three verification queries provide evidence for the main data-contract assumptions:

- The grain check tests for duplicate client × content × report-date combinations.
- The row-count and date-range check verifies the observed February 2026 feature window.
- The availability check measures Google Search Console coverage using `IS TRUE`.

These checks support aggregating the observed February daily data to one row per client × content item for the February 28, 2026 decision point.

## 4. Build the first five features

I will build five features using information available by the February 28, 2026 decision point.

The features are:

1. February clicks
2. February impressions
3. February average position
4. February content age
5. February active days

The modeling grain will be one row per client × content item.

### Features 1–3 — February Search Console performance

The first three features summarize measured Google Search Console performance during February 2026.

Only observations where `gsc_data_available IS TRUE` are included.

The dataset provides the already-aggregated Google Search Console average position as `gsc_avg_position`, so I use that field directly.

In [60]:
# Build the first three February features

feb_agg = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS feb_clicks,
    SUM(gsc_impressions) AS feb_impressions,
    AVG(gsc_avg_position) AS feb_avg_position
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

feb_agg.head()

,client_hash_id,content_hash_id,feb_clicks,feb_impressions,feb_avg_position
0,client_62f4a7e64f5e0096,content_e3bbc18f354ba564,0.0,70.0,7.564827
1,client_62f4a7e64f5e0096,content_3705d3fe869704b1,1.0,817.0,1.480969
2,client_62f4a7e64f5e0096,content_7ecb3419d7ca0c84,1.0,1235.0,4.949239
3,client_62f4a7e64f5e0096,content_8d6f91ae01ad4cd9,0.0,216.0,5.633971
4,client_9958f0a7ae1df715,content_d28cf5bc4b38320f,1.0,77.0,44.591358


### Feature 4 — February content age

The fourth feature is the age of the content as of February 28, 2026.

Before calculating it, I inspect the content dimension to identify the actual publication-date field.

In [61]:
content_schema = con.sql(f"""
DESCRIBE
SELECT *
FROM {DIM}
""").df()

content_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [62]:
# Find columns containing date-related names

date_columns = [
    column
    for column in content_schema["column_name"].tolist()
    if any(
        word in column.lower()
        for word in ["date", "publish", "created"]
    )
]

date_columns

['content_created_date',
 'content_updated_date',
 'keyword_created_date',
 'last_optimized_date',
 'optimization_eligible_date',
 'is_published']

### Feature 4 — February content age

Content age measures how many days old the content was at the February 28, 2026 decision point.

I use `content_created_date` because the creation date is historical information that can be known by the decision point.

I do not use later event fields such as `last_optimized_date`, because those could contain information that was not available at the decision point.

In [63]:
# Build February content age

content_age = con.sql(f"""
SELECT
    content_hash_id,
    DATE_DIFF(
        'day',
        CAST(content_created_date AS DATE),
        DATE '2026-02-28'
    ) AS feb_content_age
FROM {DIM}
""").df()

content_age.head()

,content_hash_id,feb_content_age
0,content_004de9653278b5a4,-91
1,content_00dc5efae381b2ab,-104
2,content_01410f2556c327ac,-70
3,content_019f27f634053ca7,-107
4,content_01efa71faea45dcc,-82


### Content-age check

The resulting feature represents the observed age of each content item in days as of February 28, 2026.

In [64]:
content_age.describe()

,feb_content_age
count,519606.000000
mean,175.350263
std,175.963027
min,-128.000000
25%,18.000000
50%,186.000000
75%,299.000000
max,500.000000


In [65]:
[
    column
    for column in schema["column_name"].tolist()
    if any(
        word in column.lower()
        for word in ["date", "day", "time"]
    )
]

['report_date']

### Feature 5 — February active days

February active days is the number of distinct February reporting days with available Google Search Console observations for each client × content item.

This feature uses only February data, so it is knowable at the February 28, 2026 decision point.

In [66]:
# Build February active days

feb_active_days = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(DISTINCT report_date) AS feb_active_days
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

feb_active_days.head()

,client_hash_id,content_hash_id,feb_active_days
0,client_e547b89c05043229,content_2a896bf67c6373e6,26
1,client_62f4a7e64f5e0096,content_db26cd04aed2a893,28
2,client_62f4a7e64f5e0096,content_62ed7761e7068c62,28
3,client_62f4a7e64f5e0096,content_334073c28476e525,20
4,client_fef1a8f436438636,content_9fdb62cc20b2a5d9,28


### Combine the five February features

I combine the February Search Console aggregates, content age, and active-day count into one feature frame.

The resulting modeling grain is one row per client × content item.

All five features are based on information available by the February 28, 2026 decision point.

In [67]:
# Combine the five February features

features = (
    feb_agg
    .merge(
        content_age,
        on="content_hash_id",
        how="left"
    )
    .merge(
        feb_active_days,
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
)

feature_columns = [
    "feb_clicks",
    "feb_impressions",
    "feb_avg_position",
    "feb_content_age",
    "feb_active_days"
]

feature_frame = features[
    [
        "client_hash_id",
        "content_hash_id"
    ] + feature_columns
].copy()

feature_frame.head()

,client_hash_id,content_hash_id,feb_clicks,feb_impressions,feb_avg_position,feb_content_age,feb_active_days
0,client_62f4a7e64f5e0096,content_e3bbc18f354ba564,0.0,70.0,7.564827,32,22
1,client_62f4a7e64f5e0096,content_3705d3fe869704b1,1.0,817.0,1.480969,32,28
2,client_62f4a7e64f5e0096,content_7ecb3419d7ca0c84,1.0,1235.0,4.949239,32,28
3,client_62f4a7e64f5e0096,content_8d6f91ae01ad4cd9,0.0,216.0,5.633971,32,28
4,client_9958f0a7ae1df715,content_d28cf5bc4b38320f,1.0,77.0,44.591358,337,27


### Feature-frame check

The feature frame should contain the client and content identifiers together with the five February-only features.

The final five features are:

- `feb_clicks`
- `feb_impressions`
- `feb_avg_position`
- `feb_content_age`
- `feb_active_days`

In [68]:
print("Number of feature rows:", len(feature_frame))

print("\nFeature columns:")
for column in feature_columns:
    print("-", column)

Number of feature rows: 153559

Feature columns:
- feb_clicks
- feb_impressions
- feb_avg_position
- feb_content_age
- feb_active_days


### Why each feature is knowable at the decision moment

- **February clicks:** knowable because they were observed during February before the February 28 decision point.
- **February impressions:** knowable because they were observed during February before the decision point.
- **February average position:** knowable because it is calculated only from February observations.
- **February content age:** knowable because `content_created_date` is historical information available by the decision point.
- **February active days:** knowable because it is calculated only from February observations with available Search Console data.

No March performance information is used to construct these five features.

## 5. Build the future outcome

March 2026 is the outcome window and occurs after the February 28, 2026 decision point.

I use March Search Console clicks only to construct the future outcome label.

I also count the number of measured March reporting days so that unavailable Search Console data is not incorrectly interpreted as zero clicks.

In [69]:
# Build the March outcome

march_label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_clicks,
    COUNT(DISTINCT report_date) AS march_measured_days
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

march_label.head()

,client_hash_id,content_hash_id,march_clicks,march_measured_days
0,client_62f4a7e64f5e0096,content_c233b46116244a98,0.0,11
1,client_62f4a7e64f5e0096,content_c60d915ee660789e,1.0,31
2,client_62f4a7e64f5e0096,content_2df51e0cfad4523f,0.0,31
3,client_62f4a7e64f5e0096,content_73da27afee28166c,1.0,31
4,client_62f4a7e64f5e0096,content_e81a683457e0c04a,2.0,31


### Create the March zero-click label

I join the future March outcome to the February feature frame using the client and content identifiers.

Only client × content items with at least one measured March day are retained.

This prevents completely unavailable March data from being treated as zero clicks.

In [70]:
# Join February features with the measured March outcome

frame = feature_frame.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

frame = frame[
    frame["march_measured_days"] > 0
].copy()

frame["march_zero_clicks"] = (
    frame["march_clicks"] == 0
).astype(int)

frame.head()

,client_hash_id,content_hash_id,feb_clicks,feb_impressions,feb_avg_position,feb_content_age,feb_active_days,march_clicks,march_measured_days,march_zero_clicks
0,client_62f4a7e64f5e0096,content_e3bbc18f354ba564,0.0,70.0,7.564827,32,22,0.0,18,1
1,client_62f4a7e64f5e0096,content_3705d3fe869704b1,1.0,817.0,1.480969,32,28,4.0,31,0
2,client_62f4a7e64f5e0096,content_7ecb3419d7ca0c84,1.0,1235.0,4.949239,32,28,2.0,31,0
3,client_62f4a7e64f5e0096,content_8d6f91ae01ad4cd9,0.0,216.0,5.633971,32,28,1.0,30,0
4,client_9958f0a7ae1df715,content_d28cf5bc4b38320f,1.0,77.0,44.591358,337,27,0.0,27,1


### Label definition

The final prediction label is `march_zero_clicks`.

- `1` means the content item had zero measured Google Search Console clicks during March 2026.
- `0` means the content item had at least one measured Google Search Console click during March 2026.

March is strictly after the February 28 decision point, so March performance is not used as an input feature.

In [71]:
label_counts = (
    frame["march_zero_clicks"]
    .value_counts()
    .sort_index()
)

label_counts

march_zero_clicks
0    57943
1    76295
Name: count, dtype: int64

In [72]:
label_percentages = (
    frame["march_zero_clicks"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

label_percentages

march_zero_clicks
0    43.16
1    56.84
Name: proportion, dtype: float64

## 6. Deliberate label leakage experiment

To demonstrate label leakage, I intentionally add `march_clicks` to the feature set.

This is deliberately incorrect because March clicks are part of the future outcome window and would not be available at the February 28, 2026 decision point.

The purpose is to demonstrate how future information can make model performance appear artificially strong.

In [73]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

### Honest model

The honest model uses only the five February features.

These are the variables that would actually be available at the February 28, 2026 decision point.

In [74]:
honest_features = [
    "feb_clicks",
    "feb_impressions",
    "feb_avg_position",
    "feb_content_age",
    "feb_active_days"
]

X = frame[honest_features].copy()
y = frame["march_zero_clicks"]

X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(
    y_test,
    honest_pred
)

print(
    "Honest model accuracy:",
    round(honest_score, 4)
)

Honest model accuracy: 0.8242


### Deliberately leaked model

I now add `march_clicks` to the five honest February features.

This is invalid because March clicks are future information relative to the February 28 decision point.

The resulting score is used only to demonstrate the effect of leakage.

In [75]:
leaked_features = honest_features + [
    "march_clicks"
]

X_leak = frame[leaked_features].copy()

X_leak = X_leak.fillna(0)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

leak_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

leak_model.fit(
    X_train_leak,
    y_train_leak
)

leak_pred = leak_model.predict(X_test_leak)

leak_score = accuracy_score(
    y_test_leak,
    leak_pred
)

print("Honest accuracy:", round(honest_score, 4))
print("Leaked accuracy:", round(leak_score, 4))
print(
    "Accuracy jump:",
    round(leak_score - honest_score, 4)
)

Honest accuracy: 0.8242
Leaked accuracy: 1.0
Accuracy jump: 0.1758


### Leakage conclusion

The leaked model includes `march_clicks`, which belongs to the future March outcome window.

The leaked model is intentionally invalid. Any improvement in its score demonstrates that future outcome information can artificially improve apparent model performance.

I therefore remove `march_clicks` from the final feature set.

The honest February-only model result is retained as the valid result for this exercise.

### Final feature set after removing leakage

After the leakage experiment, I remove `march_clicks`.

The final feature set contains only variables that are knowable at the February 28, 2026 decision point.

In [76]:
final_feature_columns = [
    "feb_clicks",
    "feb_impressions",
    "feb_avg_position",
    "feb_content_age",
    "feb_active_days"
]

print("Final feature set:")

for feature in final_feature_columns:
    print("-", feature)

Final feature set:
- feb_clicks
- feb_impressions
- feb_avg_position
- feb_content_age
- feb_active_days


## 7. Final modeling contract

### Unit of analysis

One row represents one content item for one client at the February 28, 2026 decision point.

### Feature window

February 2026.

### Final features

- `feb_clicks`
- `feb_impressions`
- `feb_avg_position`
- `feb_content_age`
- `feb_active_days`

### Outcome window

March 2026.

### Prediction target

`march_zero_clicks`, indicating whether measured March 2026 clicks are zero.

### Context

- `client_hash_id`
- `content_hash_id`
- Decision date: February 28, 2026

### Excluded

March performance variables are excluded from the final feature set because they are future information relative to the decision point.

Client names, URLs, raw queries, and other identifying information are excluded from notebook outputs.

## 8. Data limits

### Named limitation — Unbalanced history

The available history is not equally deep for every client and content item.

Some content items may have fewer observed days or less Google Search Console coverage than others. Therefore, comparisons should be treated as **observed and directional**, rather than perfectly comparable across all content.

This dataset can support predictive or decision-support analysis, but it cannot by itself establish that a feature caused a future change in clicks.

### Google Search Console coverage

Google Search Console data is not guaranteed to be available for every day. The analysis therefore distinguishes measured observations from unavailable observations instead of assuming that unavailable data means zero performance.

### Window limitation

The features are restricted to February 2026 and the outcome to March 2026. Therefore, the findings describe this particular decision window and should not automatically be assumed to hold for every future month.

In [77]:
print("Number of modeling rows:", len(frame))

print(
    "Number of zero-click outcomes:",
    int(frame["march_zero_clicks"].sum())
)

print(
    "Zero-click outcome rate:",
    round(
        frame["march_zero_clicks"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Honest model accuracy:",
    round(honest_score, 4)
)

Number of modeling rows: 134238
Number of zero-click outcomes: 76295
Zero-click outcome rate: 56.84 %
Honest model accuracy: 0.8242


## Self-check

Before submitting, confirm each item honestly:

- [ ] Every section is filled with Markdown reasoning and supporting code.
- [ ] The notebook runs from top to bottom without errors.
- [ ] The source grain was checked.
- [ ] The February row count and date range were checked.
- [ ] Google Search Console availability was checked using `IS TRUE`.
- [ ] Exactly three verification queries are used in Section 3.
- [ ] Five February-only features were created.
- [ ] Every feature has a "knowable at the decision moment because..." explanation.
- [ ] March is used only to define the future outcome.
- [ ] Unavailable March data is not automatically treated as zero clicks.
- [ ] A deliberately leaked feature was added for the leakage experiment.
- [ ] The leaked feature was removed from the final feature set.
- [ ] No client names, URLs, raw queries, or private identifying information are included.
- [ ] Claims use careful language such as observed, measured, directional, and decision-support.
- [ ] The notebook has been executed successfully from top to bottom.
- [ ] The completed notebook is committed under `work/notebooks/w03_data_contract.ipynb`.